# 評価（RAGAS で手法を比較）

- `rag/` の各手法（`answer(question)`）を回し、RAGAS で採点して比較表にする。
- **前処理は各モジュールの import 時に自動実行**（DB読み込み・モデル準備）。
- このノートは**プロジェクトのルート**で実行すること（`rag/` と `chroma_db`/`data` のパスが解決できる）。

In [ ]:
import sys, types

_m = types.ModuleType("langchain_community.chat_models.vertexai"); _m.ChatVertexAI = object
sys.modules["langchain_community.chat_models.vertexai"] = _m
import langchain_community.llms as _llms
if not hasattr(_llms, "VertexAI"): _llms.VertexAI = object

sys.path.insert(0, "rag")

from dotenv import load_dotenv; load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, LLMContextRecall
from ragas.run_config import RunConfig

judge = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite"))
METRICS = [Faithfulness(), LLMContextRecall()]

In [ ]:
eval_dataset = [
    {"question": "青色申告特別控除の最高額はいくらで、その要件は何ですか？",
     "reference": "最高65万円です。55万円の要件（事業所得等・複式簿記・期限内申告）に加え、電子帳簿保存またはe-Taxによる電子申告を行う場合に65万円。簡易な記帳の場合は10万円です。"},
    {"question": "消費税の納税義務が免除されるのはどんな場合ですか？",
     "reference": "基準期間（個人事業者はその年の前々年）の課税売上高が1,000万円以下の場合、原則として納税義務が免除されます。ただし適格請求書発行事業者の登録を受けている場合などは免除されません。"},
    {"question": "白色申告の事業専従者控除の上限額はいくらですか？",
     "reference": "配偶者は最高86万円、15歳以上のその他の親族は最高50万円を必要経費として差し引くことができます。"},
    {"question": "青色申告ができるのはどの所得がある人ですか？",
     "reference": "不動産所得、事業所得、山林所得のある人です。"},
    {"question": "取得価額がいくら未満なら減価償却せずに全額を必要経費にできますか？",
     "reference": "取得価額が10万円未満のもの、または使用可能期間が1年未満のものは、取得した年に全額を必要経費にできます。"},
    {"question": "所得税の税率は何段階で、何パーセントから何パーセントですか？",
     "reference": "所得税の税率は、5パーセントから45パーセントの7段階に区分されています。"},
    {"question": "社会保険料控除で控除できる金額はいくらですか？",
     "reference": "その年に実際に支払った金額、または給与や公的年金から差し引かれた金額の全額を控除できます。"},
    {"question": "所得税の確定申告は、いつからいつまでの所得を計算する手続きですか？",
     "reference": "毎年1月1日から12月31日までの1年間に生じた所得の金額とそれに対する所得税等の額を計算して確定させる手続きです。"},
    {"question": "取得価額が10万円以上20万円未満の減価償却資産は、どのように必要経費にできますか？",
     "reference": "一定の要件の下で、一括してその取得価額の合計額の3分の1に相当する金額を、業務の用に供した年以後3年間の各年分の必要経費に算入できます（一括償却資産）。"},
    {"question": "必要経費が「その年の必要経費」として認められる債務確定の3つの要件は何ですか？",
     "reference": "その年の12月31日までに、（1）債務が成立していること、（2）その債務に基づき具体的な給付をすべき原因となる事実が発生していること、（3）金額が合理的に算定できること、の3つをすべて満たす場合です。"},
    {"question": "基礎控除の金額は何によって決まりますか？",
     "reference": "納税者本人の合計所得金額に応じて金額が変わります。合計所得金額が多いほど控除額は小さくなり、一定額を超えると0円になります。"},
    {"question": "給与所得者が確定申告をしなくてもよいのはどんな場合ですか？",
     "reference": "給与の収入金額が2,000万円以下で、給与を1か所から受けており、給与所得・退職所得以外の所得金額が20万円以下である場合などは、確定申告をしなくてもよいことになっています。"},
]
print("質問数:", len(eval_dataset))

In [ ]:
import rag_vector, rag_hybrid, rag_rerank, rag_semantic, rag_crag, rag_agentic, rag_graphrag, rag_graph_hybrid

METHODS = {
    "vector":       rag_vector,
    "hybrid":       rag_hybrid,
    "rerank":       rag_rerank,
    "semantic":     rag_semantic,
    "crag":         rag_crag,
    "agentic":      rag_agentic,
    "graphrag":     rag_graphrag,
    "graph_hybrid": rag_graph_hybrid,
}

results = {}
for name, mod in METHODS.items():
    samples = []
    for item in eval_dataset:
        out = mod.answer(item["question"])
        samples.append(SingleTurnSample(
            user_input=item["question"],
            retrieved_contexts=out["contexts"],
            response=out["answer"],
            reference=item["reference"],
        ))
    results[name] = evaluate(EvaluationDataset(samples=samples),
                             metrics=METRICS, llm=judge,
                             run_config=RunConfig(max_workers=1))
    print(name, "done")

In [ ]:
import pandas as pd
names = [m.name for m in METRICS]
pd.DataFrame({name: res.to_pandas()[names].mean() for name, res in results.items()}).T